# h=0 nowcast sanity check

This notebook uses one `TsgamForecastEstimator` to produce the aligned nowcast and direct forecasts. Every metric uses the same valid test origins, so the comparison tests alignment rather than sample availability.

**API boundary:** `horizon` is the maximum requested horizon. Prediction output contains `horizon_0` through `horizon_H`, with `horizon_0` serving as the aligned nowcast baseline.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from tsgam_estimator import (
    TsgamEstimatorConfig,
    TsgamForecastConfig,
    TsgamForecastEstimator,
    TsgamLinearConfig,
    TsgamMultiPeriodicConfig,
    TsgamSolverConfig,
    forecast_to_long_dataframe,
    plot_forecast_origin,
)

sns.set_theme(style="whitegrid", context="notebook")

SEED = 42
N_SAMPLES = 504
TRAIN_SAMPLES = 336
HORIZON = 12
DRIVER_PHI = 0.72
DRIVER_INNOVATION_SCALE = 0.55
NOISE_SCALE = 0.05


## Synthetic signal

The target combines exactly representable daily and weekly Fourier structure with a distributed linear driver effect:

$$y_t = 8 + p_t + 1.8x_t + 0.65x_{t-1} + \epsilon_t.$$

The seeded AR(1) driver makes nearby horizons related while preserving future innovations that are unknown at the origin. The nowcast sees both $x_t$ and $x_{t-1}$; forecasts must infer the future contemporaneous driver term.

In [ ]:
def make_synthetic_data() -> pd.DataFrame:
    rng = np.random.default_rng(SEED)
    sample_ix = np.arange(N_SAMPLES, dtype=float)
    timestamps = pd.date_range("2024-01-01", periods=N_SAMPLES, freq="1h")

    innovations = rng.normal(0.0, DRIVER_INNOVATION_SCALE, size=N_SAMPLES)
    driver = np.zeros(N_SAMPLES)
    for ix in range(1, N_SAMPLES):
        driver[ix] = DRIVER_PHI * driver[ix - 1] + innovations[ix]

    previous_driver = np.concatenate(([driver[0]], driver[:-1]))
    periodic = (
        1.4 * np.sin(2.0 * np.pi * sample_ix / 24.0)
        + 0.45 * np.cos(4.0 * np.pi * sample_ix / 24.0)
        + 0.6 * np.cos(2.0 * np.pi * sample_ix / 168.0)
    )
    noise = rng.normal(0.0, NOISE_SCALE, size=N_SAMPLES)
    current_driver_component = 1.8 * driver
    lagged_driver_component = 0.65 * previous_driver
    target = (
        8.0
        + periodic
        + current_driver_component
        + lagged_driver_component
        + noise
    )
    return pd.DataFrame(
        {
            "target": target,
            "driver": driver,
            "periodic": periodic,
            "current_driver_component": current_driver_component,
            "lagged_driver_component": lagged_driver_component,
            "noise": noise,
        },
        index=timestamps,
    )


data = make_synthetic_data()
display(data.head(4).round(3))


In [ ]:
overview = data.iloc[-240:]
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, constrained_layout=True)
axes[0].plot(overview.index, overview["target"], color="#202124", linewidth=1.5, label="observed target")
axes[0].plot(overview.index, 8.0 + overview["periodic"], color="#4C78A8", linewidth=1.2, label="periodic baseline")
axes[0].axvline(data.index[TRAIN_SAMPLES], color="#D1495B", linestyle="--", linewidth=1.4, label="test starts")
axes[0].set_ylabel("target")
axes[0].set_title("Synthetic target and held-out evaluation region", loc="left")
axes[0].legend(ncol=3, frameon=False, loc="upper right")

axes[1].plot(overview.index, overview["driver"], color="#2A9D8F", linewidth=1.2)
axes[1].axvline(data.index[TRAIN_SAMPLES], color="#D1495B", linestyle="--", linewidth=1.4)
axes[1].set_ylabel("driver x_t")
axes[1].set_xlabel("timestamp")
axes[1].set_title("Origin-known driver with future AR innovations", loc="left")
plt.show()


## Fit the diagnostic and forecast models

Both models use the same training cutoff and base configuration. In this codebase, exogenous offsets `[-1, 0]` supply the previous and current rows. The direct forecaster shifts those origin rows to each target timestamp internally.

In [ ]:
def make_base_config() -> TsgamEstimatorConfig:
    return TsgamEstimatorConfig(
        multi_periodic_config=TsgamMultiPeriodicConfig(
            num_harmonics=[2, 1],
            periods=[24, 168],
            reg_weight=1.0e-7,
        ),
        exog_config=[
            TsgamLinearConfig(
                lags=[-1, 0],
                reg_weight=1.0e-7,
                diff_reg_weight=0.0,
            )
        ],
        solver_config=TsgamSolverConfig(solver="CLARABEL", verbose=False),
    )


X = data[["driver"]]
y = data["target"]
X_train = X.iloc[:TRAIN_SAMPLES]
y_train = y.iloc[:TRAIN_SAMPLES].to_numpy()

forecast_model = TsgamForecastEstimator(
    TsgamForecastConfig(
        horizon=HORIZON,
        base_config=make_base_config(),
        mode="independent",
    )
).fit(X_train, y_train)

assert forecast_model.horizons_ == list(range(HORIZON + 1))


## Exact origin and target alignment

The final `HORIZON` rows cannot be common origins because their longest-horizon targets do not exist. One earlier row is passed as lag history, but it is not scored. Predictions remain indexed by origin time; actuals are selected at `origin_time + h`.

In [ ]:
common_origins = data.index[TRAIN_SAMPLES : N_SAMPLES - HORIZON]
prediction_X = X.iloc[TRAIN_SAMPLES - 1 : N_SAMPLES - HORIZON]

forecast_predictions = forecast_model.predict(prediction_X).reindex(common_origins)

assert list(forecast_predictions.columns) == [
    f"horizon_{h}" for h in range(HORIZON + 1)
]
assert forecast_predictions.notna().all().all()

evaluation = forecast_to_long_dataframe(
    forecast_predictions,
    actual=y,
    freq="1h",
    model="TsgamForecastEstimator",
)
evaluation["error"] = evaluation["prediction"] - evaluation["actual"]
evaluation["target offset (hours)"] = (
    (evaluation["target_time"] - evaluation["origin_time"])
    / pd.Timedelta(hours=1)
).astype(int)

origin_counts = evaluation.groupby("horizon")["origin_time"].nunique()
assert origin_counts.eq(len(common_origins)).all()
assert evaluation["target offset (hours)"].eq(evaluation["horizon"]).all()

first_origin = common_origins[0]
alignment_table = evaluation.loc[
    evaluation["origin_time"].eq(first_origin),
    ["model", "origin_time", "horizon", "target_time", "target offset (hours)", "prediction", "actual"],
].copy()
alignment_table["known x_t"] = data.loc[first_origin, "driver"]
alignment_table["known x_t-1"] = data.loc[first_origin - pd.Timedelta(hours=1), "driver"]
display(alignment_table.round({"prediction": 3, "actual": 3, "known x_t": 3, "known x_t-1": 3}))


## Horizon metrics

The table and chart begin at `h=0`. The sharp gap after the nowcast is intentional: future driver innovations are unavailable at the origin, while periodic structure remains forecastable.

In [ ]:
metrics = (
    evaluation.groupby(["horizon", "model"], as_index=False)
    .agg(
        rmse=("error", lambda error: float(np.sqrt(np.mean(np.square(error))))),
        mae=("error", lambda error: float(np.mean(np.abs(error)))),
        origins=("origin_time", "nunique"),
    )
    .sort_values("horizon")
    .reset_index(drop=True)
)

display(
    metrics.style
    .format({"rmse": "{:.4f}", "mae": "{:.4f}"})
    .highlight_min(subset=["rmse", "mae"], color="#F4A6A0")
)

h0_rmse = metrics.loc[metrics["horizon"].eq(0), "rmse"].item()
forecast_rmse_floor = metrics.loc[metrics["horizon"].gt(0), "rmse"].min()
assert h0_rmse < 0.25 * forecast_rmse_floor

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, metric, label in zip(axes, ["rmse", "mae"], ["RMSE", "MAE"], strict=True):
    ax.plot(metrics["horizon"], metrics[metric], color="#2A9D8F", marker="o", linewidth=2.0)
    h0_value = metrics.loc[metrics["horizon"].eq(0), metric].item()
    ax.scatter([0], [h0_value], color="#D1495B", s=85, zorder=3, label="h=0 diagnostic")
    ax.axvline(0.5, color="#6B7280", linestyle="--", linewidth=1.0)
    ax.set_xticks(range(HORIZON + 1))
    ax.set_xlabel("horizon (hours)")
    ax.set_ylabel(label)
    ax.set_title(f"{label} on {len(common_origins)} common origins", loc="left")
    ax.legend(frameon=False, loc="lower right")
plt.show()


## Forecast examples

Each panel shows observed history, the fitted h=0 value at the forecast origin, the `h=1..H` path, and the realized target on one target-time axis.

In [ ]:
example_origins = common_origins[[0, len(common_origins) // 2, -1]]
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharey=True, constrained_layout=True)

for ax, origin in zip(axes, example_origins, strict=True):
    plot_forecast_origin(
        forecast_predictions,
        actual=y,
        origin=origin,
        history_steps=12,
        freq="1h",
        ax=ax,
    )
    ax.set_title(f"Origin {origin:%Y-%m-%d %H:%M}", loc="left", fontsize=11)
    ax.set_ylabel("target")
plt.show()


## Takeaway

The low `h=0` error confirms that the base regression can recover the target relationship at the origin. The larger future errors are expected because future driver innovations are unavailable. If `h=0` were not the minimum in this controlled setup, origin/target alignment and feature availability should be investigated before interpreting forecast quality.

This diagnostic does not add `horizon_0` to `TsgamForecastEstimator`; it deliberately keeps nowcasting and forecasting as separate model contracts.